# Week 3: NLP Model Comparison

## Objective
Compare two approaches for movie review sentiment classification:

1. TF-IDF + Logistic Regression
2. Pretrained DistilBERT Transformer

The goal is to evaluate their accuracy, performance, speed,
interpretability, and ability to understand different types of text.

In [21]:
# Import required libraries

import pandas as pd
import numpy as np
import time
import re
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("All basic libraries imported successfully!")

All basic libraries imported successfully!


## 1. Loading the IMDB Dataset

We use the IMDB movie review dataset, containing:
- 25,000 training reviews
- 25,000 testing reviews
- Binary sentiment labels:
    - 1 = Positive
    - 0 = Negative

In [22]:
# Load the IMDB training and testing datasets

train_url = "https://huggingface.co/datasets/stanfordnlp/imdb/resolve/main/plain_text/train-00000-of-00001.parquet"

test_url = "https://huggingface.co/datasets/stanfordnlp/imdb/resolve/main/plain_text/test-00000-of-00001.parquet"

train = pd.read_parquet(train_url)
test = pd.read_parquet(test_url)

print("Training dataset shape:", train.shape)
print("Testing dataset shape:", test.shape)

print("\nTraining dataset columns:")
print(train.columns.tolist())

print("\nFirst five training rows:")
display(train.head())

Training dataset shape: (25000, 2)
Testing dataset shape: (25000, 2)

Training dataset columns:
['text', 'label']

First five training rows:


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


## 2. Exploratory Data Analysis

Before training our models, we inspect the dataset to understand
its sentiment distribution, review lengths, and HTML artifacts.

In [23]:
# 1. Check label balance

print("TRAINING LABEL DISTRIBUTION")
print(train["label"].value_counts().sort_index())

print("\nTESTING LABEL DISTRIBUTION")
print(test["label"].value_counts().sort_index())


# 2. Print three movie reviews

print("\nTHREE SAMPLE MOVIE REVIEWS")

for i in range(3):
    print(f"\nReview {i + 1}:")
    print(train.iloc[i]["text"])
    print("Label:", train.iloc[i]["label"])


# 3. Calculate median review length in words

train["review_length"] = train["text"].str.split().str.len()

median_length = train["review_length"].median()

print("\nMedian review length:", median_length, "words")


# 4. Count reviews containing <br />

br_count = train["text"].str.contains(
    "<br />",
    regex=False,
    na=False
).sum()

print("Training reviews containing <br />:", br_count)

TRAINING LABEL DISTRIBUTION
label
0    12500
1    12500
Name: count, dtype: int64

TESTING LABEL DISTRIBUTION
label
0    12500
1    12500
Name: count, dtype: int64

THREE SAMPLE MOVIE REVIEWS

Review 1:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teac

## 3. Baseline Model

A baseline is a simple reference point used to evaluate whether
our machine-learning models are actually learning useful patterns.

We calculate the accuracy of always predicting class 0 or class 1.

In [24]:
# Calculate baseline accuracy

total_reviews = len(test)

negative_predictions = np.zeros(total_reviews)
positive_predictions = np.ones(total_reviews)

# Always predict negative (0)
baseline_negative_accuracy = accuracy_score(
    test["label"], negative_predictions
)

# Always predict positive (1)
baseline_positive_accuracy = accuracy_score(
    test["label"], positive_predictions
)

print("Baseline: Always predicting NEGATIVE (0)")
print(f"Accuracy: {baseline_negative_accuracy:.4f}")

print("\nBaseline: Always predicting POSITIVE (1)")
print(f"Accuracy: {baseline_positive_accuracy:.4f}")

Baseline: Always predicting NEGATIVE (0)
Accuracy: 0.5000

Baseline: Always predicting POSITIVE (1)
Accuracy: 0.5000


## 4. Model A: TF-IDF + Logistic Regression

We build a single Scikit-learn Pipeline containing:

1. TfidfVectorizer() to convert text into numerical features.
2. LogisticRegression(max_iter=1000) to learn sentiment patterns.

We train on all 25,000 training reviews and evaluate on all
25,000 testing reviews.

In [25]:
# Create Model A pipeline

model_a = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Separate features and labels
X_train = train["text"]
y_train = train["label"]

X_test = test["text"]
y_test = test["label"]

# Train the model and measure training time
start_time = time.time()

model_a.fit(X_train, y_train)

training_time_a = time.time() - start_time

# Make predictions
predictions_a = model_a.predict(X_test)

# Evaluate the model
accuracy_a = accuracy_score(y_test, predictions_a)
f1_a = f1_score(y_test, predictions_a)
confusion_a = confusion_matrix(y_test, predictions_a)

# Display results
print("MODEL A: TF-IDF + LOGISTIC REGRESSION")
print("-" * 45)

print(f"Training time: {training_time_a:.2f} seconds")
print(f"Accuracy: {accuracy_a:.4f}")
print(f"F1 Score: {f1_a:.4f}")

print("\nConfusion Matrix:")
print(confusion_a)

print("\nClassification Report:")
print(classification_report(y_test, predictions_a))

MODEL A: TF-IDF + LOGISTIC REGRESSION
---------------------------------------------
Training time: 10.30 seconds
Accuracy: 0.8829
F1 Score: 0.8827

Confusion Matrix:
[[11055  1445]
 [ 1482 11018]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.88      0.88     12500
           1       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [26]:
# Create Model A pipeline

model_a = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Separate features and labels
X_train = train["text"]
y_train = train["label"]

X_test = test["text"]
y_test = test["label"]

# Train the model and measure training time
start_time = time.time()

model_a.fit(X_train, y_train)

training_time_a = time.time() - start_time

# Make predictions
predictions_a = model_a.predict(X_test)

# Evaluate the model
accuracy_a = accuracy_score(y_test, predictions_a)
f1_a = f1_score(y_test, predictions_a)
confusion_a = confusion_matrix(y_test, predictions_a)

# Display results
print("MODEL A: TF-IDF + LOGISTIC REGRESSION")
print("-" * 45)

print(f"Training time: {training_time_a:.2f} seconds")
print(f"Accuracy: {accuracy_a:.4f}")
print(f"F1 Score: {f1_a:.4f}")

print("\nConfusion Matrix:")
print(confusion_a)

print("\nClassification Report:")
print(classification_report(y_test, predictions_a))



MODEL A: TF-IDF + LOGISTIC REGRESSION
---------------------------------------------
Training time: 11.67 seconds
Accuracy: 0.8829
F1 Score: 0.8827

Confusion Matrix:
[[11055  1445]
 [ 1482 11018]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.88      0.88     12500
           1       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



## 5. Model A Experiment: Unigrams vs Bigrams

We experiment with:

- ngram_range=(1, 2): Uses individual words and two-word combinations.
- min_df=2: Ignores words and phrases appearing in fewer than two documents.

We compare the new model against our original TF-IDF model.

In [27]:
# Create a new pipeline using unigrams and bigrams

model_a_bigram = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2
    )),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train and measure training time

start_time = time.time()

model_a_bigram.fit(X_train, y_train)

training_time_bigram = time.time() - start_time

# Predict on the complete test dataset

predictions_bigram = model_a_bigram.predict(X_test)

# Evaluate the bigram model

accuracy_bigram = accuracy_score(y_test, predictions_bigram)
f1_bigram = f1_score(y_test, predictions_bigram)

# Calculate improvement over original Model A

accuracy_difference = accuracy_bigram - accuracy_a
f1_difference = f1_bigram - f1_a

print("MODEL A: UNIGRAMS VS BIGRAMS")
print("-" * 45)

print(f"Original accuracy: {accuracy_a:.4f}")
print(f"Bigram accuracy:   {accuracy_bigram:.4f}")

print(f"\nOriginal F1 score: {f1_a:.4f}")
print(f"Bigram F1 score:   {f1_bigram:.4f}")

print(f"\nAccuracy difference: {accuracy_difference:+.4f}")
print(f"F1 difference:       {f1_difference:+.4f}")

print(f"\nBigram training time: {training_time_bigram:.2f} seconds")

if accuracy_difference > 0:
    print("\nResult: Bigrams improved accuracy.")
elif accuracy_difference < 0:
    print("\nResult: Bigrams reduced accuracy.")
else:
    print("\nResult: Both models achieved the same accuracy.")

MODEL A: UNIGRAMS VS BIGRAMS
---------------------------------------------
Original accuracy: 0.8829
Bigram accuracy:   0.8898

Original F1 score: 0.8827
Bigram F1 score:   0.8905

Accuracy difference: +0.0069
F1 difference:       +0.0077

Bigram training time: 24.93 seconds

Result: Bigrams improved accuracy.


## 6. Opening Model A: Important Words

Logistic Regression learns coefficients (weights) for each TF-IDF feature.

- Positive coefficients push predictions towards positive sentiment.
- Negative coefficients push predictions towards negative sentiment.

We inspect the 15 strongest positive and 15 strongest negative words
to understand what the model has learned.

In [28]:
# Extract vocabulary and learned coefficients

tfidf = model_a.named_steps["tfidf"]
classifier = model_a.named_steps["classifier"]

feature_names = tfidf.get_feature_names_out()
coefficients = classifier.coef_[0]

# Find the 15 strongest positive words
top_positive_indices = np.argsort(coefficients)[-15:][::-1]

# Find the 15 strongest negative words
top_negative_indices = np.argsort(coefficients)[:15]

print("TOP 15 WORDS PUSHING TOWARDS POSITIVE SENTIMENT")
print("-" * 55)

for index in top_positive_indices:
    print(f"{feature_names[index]:25s} {coefficients[index]:.4f}")


print("\nTOP 15 WORDS PUSHING TOWARDS NEGATIVE SENTIMENT")
print("-" * 55)

for index in top_negative_indices:
    print(f"{feature_names[index]:25s} {coefficients[index]:.4f}")

TOP 15 WORDS PUSHING TOWARDS POSITIVE SENTIMENT
-------------------------------------------------------
great                     7.3860
excellent                 6.3688
best                      5.0528
perfect                   4.9973
wonderful                 4.8051
amazing                   4.2911
well                      4.0117
today                     3.8903
loved                     3.8840
favorite                  3.8016
fun                       3.7983
love                      3.7348
enjoyed                   3.5714
highly                    3.4815
brilliant                 3.4626

TOP 15 WORDS PUSHING TOWARDS NEGATIVE SENTIMENT
-------------------------------------------------------
worst                     -9.4386
bad                       -7.6834
awful                     -6.6274
waste                     -6.3735
boring                    -6.1698
poor                      -5.5661
terrible                  -4.9118
nothing                   -4.7825
worse                   

One surprising word is "today". It has a positive coefficient even though it does not directly express happiness or positive emotion. This shows that the model learns associations from the training data rather than understanding the actual meaning of every word.

In [29]:
import torch
import transformers

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.15.0+cpu
Transformers version: 5.22.2
GPU available: False


## 7. Model B: Pretrained DistilBERT

In this section, we use a pretrained transformer model for sentiment analysis.

Unlike Model A, DistilBERT is already trained and fine-tuned for positive and negative sentiment classification.

We will:
- Load the pretrained DistilBERT model.
- Select a fixed sample of 1,000 test reviews.
- Test the longest review first to observe the token length error.
- Apply truncation to handle the 512-token limit.
- Evaluate the model's accuracy and F1 score.

In [30]:
# Score the same 1,000 reviews using DistilBERT

start_time = time.perf_counter()

transformer_outputs = sentiment_pipe(
    sample_texts,
    truncation=True,
    max_length=512,
    batch_size=16
)

transformer_seconds = time.perf_counter() - start_time

label_map = {"POSITIVE": 1, "NEGATIVE": 0}

predictions_b = np.array([
    label_map[item["label"]]
    for item in transformer_outputs
])

accuracy_b = accuracy_score(sample_labels, predictions_b)
f1_b = f1_score(sample_labels, predictions_b)

print("Reviews scored:", len(predictions_b))
print("Total time:", round(transformer_seconds, 2), "seconds")
print("Model B Accuracy:", round(accuracy_b, 4))
print("Model B F1:", round(f1_b, 4))

Reviews scored: 1000
Total time: 576.77 seconds
Model B Accuracy: 0.883
Model B F1: 0.878


In [31]:
from transformers import pipeline

MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME
)

# Select the required fixed sample
sample = test.sample(1000, random_state=42).copy()

sample_texts = sample["text"].tolist()
sample_labels = sample["label"].astype(int).to_numpy()

# Find the longest review
longest_index = sample["text"].str.split().str.len().idxmax()
longest_review = sample.loc[longest_index, "text"]

print("Longest review length:", len(longest_review.split()), "words")

# First attempt without truncation
try:
    print(sentiment_pipe(longest_review))

except Exception as error:
    print("Expected length-related error:")
    print(type(error).__name__, str(error)[:1200])

# Fix the error using truncation
start_time = time.perf_counter()

transformer_outputs = sentiment_pipe(
    sample_texts,
    truncation=True,
    max_length=512,
    batch_size=16
)

transformer_seconds = time.perf_counter() - start_time

# Convert sentiment labels into binary values
label_map = {
    "POSITIVE": 1,
    "NEGATIVE": 0
}

predictions_b = np.array([
    label_map[item["label"]]
    for item in transformer_outputs
])

# Evaluate Model B
accuracy_b = accuracy_score(sample_labels, predictions_b)
f1_b = f1_score(sample_labels, predictions_b)

print("Reviews scored:", len(predictions_b))
print("Total time:", round(transformer_seconds, 2), "seconds")
print("Model B Accuracy:", round(accuracy_b, 4))
print("Model B F1:", round(f1_b, 4))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1475 > 512). Running this sequence through the model will result in indexing errors


Longest review length: 1090 words
Expected length-related error:
RuntimeError The size of tensor a (1475) must match the size of tensor b (512) at non-singleton dimension 1
Reviews scored: 1000
Total time: 584.27 seconds
Model B Accuracy: 0.883
Model B F1: 0.878


## 8. Fair Fight: TF-IDF vs DistilBERT

We compare both models using exactly the same 1,000 test reviews.

We also divide the reviews into two groups:
- Reviews with 512 tokens or fewer.
- Reviews with more than 512 tokens.

This helps us understand how the transformer model's token limit affects its performance.

In [32]:
# Model A predictions on the same 1,000 reviews

start_time = time.perf_counter()

predictions_a_sample = model_a.predict(sample_texts)

model_a_sample_seconds = time.perf_counter() - start_time

# Count tokens using DistilBERT tokenizer

tokenizer = sentiment_pipe.tokenizer

token_counts = [
    len(tokenizer.encode(text, add_special_tokens=False))
    for text in sample_texts
]

is_long = np.array(token_counts) > 512

# Compare accuracy for each group

def report_subset(name, mask):

    count = int(mask.sum())

    if count == 0:
        print(name, ": No reviews found")
        return

    print(
        f"{name} (n={count}): "
        f"TF-IDF accuracy = "
        f"{accuracy_score(sample_labels[mask], predictions_a_sample[mask]):.4f}, "
        f"DistilBERT accuracy = "
        f"{accuracy_score(sample_labels[mask], predictions_b[mask]):.4f}"
    )

print("SAME SAMPLE OVERALL")

print(
    "TF-IDF accuracy:",
    round(accuracy_score(sample_labels, predictions_a_sample), 4)
)

print(
    "DistilBERT accuracy:",
    round(accuracy_b, 4)
)

print("Reviews above 512 tokens:", is_long.sum())

print("\nBY TOKEN LENGTH")

report_subset("512 tokens or fewer", ~is_long)

report_subset("More than 512 tokens", is_long)

SAME SAMPLE OVERALL
TF-IDF accuracy: 0.879
DistilBERT accuracy: 0.883
Reviews above 512 tokens: 133

BY TOKEN LENGTH
512 tokens or fewer (n=867): TF-IDF accuracy = 0.8731, DistilBERT accuracy = 0.8927
More than 512 tokens (n=133): TF-IDF accuracy = 0.9173, DistilBERT accuracy = 0.8195


## 9. Short Sentence Challenge

We test both models on short sentences containing negation, contrast, sarcasm, and neutral sentiment.

This experiment helps us understand how the models handle different language patterns.

In [33]:
challenge_sentences = [
    "This movie was not good.",
    "Not bad at all, I really enjoyed it.",
    "I expected to hate it, but I loved it.",
    "Oh great, another two hours of my life I will never get back.",
    "The acting was fine.",
    "The plot was messy, but the performances made the film worthwhile."
]

# Model A
challenge_a = model_a.predict(challenge_sentences)

# Model B
challenge_b_raw = sentiment_pipe(
    challenge_sentences,
    truncation=True,
    max_length=512
)

challenge_b = [
    label_map[item["label"]]
    for item in challenge_b_raw
]

expected = np.array([0, 1, 1, 0, np.nan, 1])

challenge_results = pd.DataFrame({
    "Sentence": challenge_sentences,

    "Expected": [
        "NEGATIVE",
        "POSITIVE",
        "POSITIVE",
        "NEGATIVE",
        "Neutral / ambiguous",
        "POSITIVE"
    ],

    "TF-IDF": [
        "POSITIVE" if p == 1 else "NEGATIVE"
        for p in challenge_a
    ],

    "DistilBERT": [
        "POSITIVE" if p == 1 else "NEGATIVE"
        for p in challenge_b
    ]
})

display(challenge_results)

# Compare accuracy on five clearly labeled sentences

scorable = ~np.isnan(expected)

print("TF-IDF accuracy:",
      accuracy_score(expected[scorable].astype(int),
                     challenge_a[scorable]))

print("DistilBERT accuracy:",
      accuracy_score(expected[scorable].astype(int),
                     np.array(challenge_b)[scorable]))

,Sentence,Expected,TF-IDF,DistilBERT
0,This movie was not good.,NEGATIVE,NEGATIVE,NEGATIVE
1,"Not bad at all, I really enjoyed it.",POSITIVE,NEGATIVE,POSITIVE
2,"I expected to hate it, but I loved it.",POSITIVE,POSITIVE,POSITIVE
3,"Oh great, another two hours of my life I will ...",NEGATIVE,POSITIVE,POSITIVE
4,The acting was fine.,Neutral / ambiguous,NEGATIVE,POSITIVE
5,"The plot was messy, but the performances made ...",POSITIVE,NEGATIVE,POSITIVE


TF-IDF accuracy: 0.4
DistilBERT accuracy: 0.8


## 10. One-Table Model Comparison

We compare both models based on:

- Accuracy on the same 1,000 reviews.
- Prediction time per review.
- Model size on disk.
- Whether the model needs IMDB training data.
- Whether an individual prediction can be explained.

In [34]:
from pathlib import Path

In [35]:
import joblib
import pandas as pd

In [36]:
# Save Model A

model_a_path = Path("tfidf_logistic_model.joblib")

joblib.dump(model_a, model_a_path)

model_a_size_mb = model_a_path.stat().st_size / (1024 ** 2)

# Measure DistilBERT cache size

hf_cache = Path.home() / ".cache" / "huggingface" / "hub"

model_cache = hf_cache / (
    "models--distilbert--distilbert-base-uncased-finetuned-sst-2-english"
)

model_b_size_mb = (
    sum(f.stat().st_size for f in model_cache.rglob("*") if f.is_file())
    / (1024 ** 2)
    if model_cache.exists()
    else float("nan")
)

# Create comparison table

comparison = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression",
        "Accuracy": accuracy_score(sample_labels, predictions_a_sample),
        "Time per review (seconds)": model_a_sample_seconds / 1000,
        "Size (MB)": model_a_size_mb,
        "Needs IMDB training data": "Yes",
        "Explain individual prediction": "Relatively easy"
    },
    {
        "Model": "DistilBERT",
        "Accuracy": accuracy_b,
        "Time per review (seconds)": transformer_seconds / 1000,
        "Size (MB)": model_b_size_mb,
        "Needs IMDB training data": "No",
        "Explain individual prediction": "Difficult"
    }
])

display(comparison.round(4))

,Model,Accuracy,Time per review (seconds),Size (MB),Needs IMDB training data,Explain individual prediction
0,TF-IDF + Logistic Regression,0.879,0.0006,2.1251,Yes,Relatively easy
1,DistilBERT,0.883,0.5843,255.6465,No,Difficult


## 11. Reflection — Answer in Your Own Words

### 1. Why does a pretrained transformer usually beat TF-IDF, and when would you still choose TF-IDF?

A pretrained transformer understands more context, word order, and relationships between words. TF-IDF mainly uses word frequencies and n-grams, so it can struggle with negation and sarcasm.

Based on my results, DistilBERT achieved 88.3% accuracy, while TF-IDF achieved 87.9% accuracy.

I would still choose TF-IDF when I need a faster, smaller, cheaper, and more interpretable model.

### 2. What is backpropagation actually doing?

Backpropagation calculates how much each model parameter contributes to the prediction error. It calculates gradients using the chain rule and sends this information backwards through the neural network. The optimizer uses these gradients to update the weights and reduce future prediction errors.